In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss, accuracy_score, confusion_matrix, classification_report
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
# 5 La Liga seasons, 2021/22 to 2025/26 ("2122" = 2021/22). More matches to learn from,
# and Elo is warmed up by the time the latest season starts.
SEASONS = ['2122', '2223', '2324', '2425', '2526']

frames = []
for season in SEASONS:
    df = pd.read_csv(f'https://www.football-data.co.uk/mmz4281/{season}/SP1.csv')
    df['season'] = season          # remember which season each match is from
    frames.append(df)

# Stack the seasons into one table; newer seasons have extra columns, older ones get blanks there.
model_data = pd.concat(frames, ignore_index=True)
model_data.head()

C:\Users\Guninder Singh\AppData\Local\Temp\ipykernel_8256\518266306.py:15: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['season'] = season          # remember which season each match is from
C:\Users\Guninder Singh\AppData\Local\Temp\ipykernel_8256\518266306.py:15: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['season'] = season          # remember which season each match is from


,Div,Date,Time,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,...,BMGMCA,BVCH,BVCD,BVCA,CLCH,CLCD,CLCA,LBCH,LBCD,LBCA
0,SP1,13/08/2021,20:00,Valencia,Getafe,1,0,H,1,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,SP1,14/08/2021,18:30,Cadiz,Levante,1,1,D,0,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,SP1,14/08/2021,18:30,Mallorca,Betis,1,1,D,1,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,SP1,14/08/2021,21:00,Alaves,Real Madrid,1,4,A,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,SP1,14/08/2021,21:00,Osasuna,Espanol,0,0,D,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
model_data.shape

(1900, 162)

In [ ]:
model_data.columns

Index(['Div', 'Date', 'Time', 'HomeTeam', 'AwayTeam', 'FTHG', 'FTAG', 'FTR',
       'HTHG', 'HTAG',
       ...
       'BMGMCA', 'BVCH', 'BVCD', 'BVCA', 'CLCH', 'CLCD', 'CLCA', 'LBCH',
       'LBCD', 'LBCA'],
      dtype='str', length=162)

In [ ]:

model_data.dtypes

Div             str
Date            str
Time            str
HomeTeam        str
AwayTeam        str
             ...   
CLCD        float64
CLCA        float64
LBCH        float64
LBCD        float64
LBCA        float64
Length: 162, dtype: object

In [ ]:
model_data.isnull().sum()

Div            0
Date           0
Time           0
HomeTeam       0
AwayTeam       0
            ... 
CLCD        1622
CLCA        1622
LBCH        1603
LBCD        1603
LBCA        1603
Length: 162, dtype: int64

In [ ]:
# Missing values per column, keeping only the columns that have any
missing = model_data.isnull().sum()
missing[missing > 0]

BWH      161
BWD      161
BWA      161
IWH      950
IWD      950
        ... 
CLCD    1622
CLCA    1622
LBCH    1603
LBCD    1603
LBCA    1603
Length: 99, dtype: int64

In [ ]:
model_data['Date'] = pd.to_datetime(model_data['Date'], format='%d/%m/%Y')


In [ ]:
# Check the conversion: real dates now, covering one season
print(model_data['Date'].dtype)
print(model_data['Date'].min(), '->', model_data['Date'].max())

# Oldest match first, rows renumbered 0..379 (drop=True so the old index doesn't become a column)
model_data = model_data.sort_values('Date').reset_index(drop=True)
model_data[['Date', 'HomeTeam', 'AwayTeam', 'FTR']].head()

datetime64[us]
2021-08-13 00:00:00 -> 2026-05-24 00:00:00


,Date,HomeTeam,AwayTeam,FTR
0,2021-08-13,Valencia,Getafe,H
1,2021-08-14,Cadiz,Levante,D
2,2021-08-14,Mallorca,Betis,D
3,2021-08-14,Alaves,Real Madrid,A
4,2021-08-14,Osasuna,Espanol,D


In [ ]:
# Match table: only the columns we use, with readable names.
# Avg* = average odds across bookmakers (complete; Pinnacle's PS* columns have gaps).
matches = model_data[['season', 'Date', 'HomeTeam', 'AwayTeam', 'FTHG', 'FTAG', 'FTR', 'AvgH', 'AvgD', 'AvgA']].copy()
matches = matches.rename(columns={
    'Date': 'date',
    'HomeTeam': 'home',
    'AwayTeam': 'away',
    'FTHG': 'home_goals',
    'FTAG': 'away_goals',
    'FTR': 'result',
    'AvgH': 'odds_home',
    'AvgD': 'odds_draw',
    'AvgA': 'odds_away',
})
matches.head()

,season,date,home,away,home_goals,away_goals,result,odds_home,odds_draw,odds_away
0,2122,2021-08-13,Valencia,Getafe,1,0,H,2.64,3.01,3.06
1,2122,2021-08-14,Cadiz,Levante,1,1,D,2.78,3.23,2.66
2,2122,2021-08-14,Mallorca,Betis,1,1,D,3.38,3.29,2.24
3,2122,2021-08-14,Alaves,Real Madrid,1,4,A,7.08,4.57,1.46
4,2122,2021-08-14,Osasuna,Espanol,0,0,D,2.29,3.26,3.31


In [ ]:
# Checks: 380 matches x 9 columns, nothing missing, and how often home / draw / away wins
print(matches.shape)
print(matches.isnull().sum().sum(), 'missing values')
matches['result'].value_counts(normalize=True).round(3)

(1900, 10)
0 missing values


result
H    0.457
A    0.281
D    0.262
Name: proportion, dtype: float64

In [ ]:
# Implied probability = 1 / odds, one per match (stored as columns in the table)
matches['imp_home'] = 1 / matches['odds_home']
matches['imp_draw'] = 1 / matches['odds_draw']
matches['imp_away'] = 1 / matches['odds_away']

# Overround = the three added up; anything above 1 is the bookmakers' cut
matches['overround'] = matches['imp_home'] + matches['imp_draw'] + matches['imp_away']
print('average overround:', round(matches['overround'].mean(), 4))

average overround: 1.0487


In [ ]:
# Task 1: fair probabilities = implied probability / overround (removes the bookmakers' cut)
matches['p_home'] = matches['imp_home'] / matches['overround']
matches['p_draw'] = matches['imp_draw'] / matches['overround']
matches['p_away'] = matches['imp_away'] / matches['overround']

matches[['home', 'away', 'odds_home', 'odds_draw', 'odds_away', 'p_home', 'p_draw', 'p_away']].head()

,home,away,odds_home,odds_draw,odds_away,p_home,p_draw,p_away
0,Valencia,Getafe,2.64,3.01,3.06,0.364987,0.320122,0.314891
1,Cadiz,Levante,2.78,3.23,2.66,0.344140,0.296195,0.359665
2,Mallorca,Betis,3.38,3.29,2.24,0.282783,0.290518,0.426699
3,Alaves,Real Madrid,7.08,4.57,1.46,0.135162,0.209397,0.655441
4,Osasuna,Espanol,2.29,3.26,3.31,0.417659,0.293386,0.288955


In [ ]:
# Task 2: the three fair probabilities must add up to 1 for every match
prob_sum = matches['p_home'] + matches['p_draw'] + matches['p_away']
print(prob_sum.min(), prob_sum.max())

0.9999999999999998 1.0000000000000002


In [ ]:
# Probability the market gave to what actually happened.
# np.select = a per-row if/elif: for each match, take the column whose condition is True.
conditions = [
    matches['result'] == 'H',
    matches['result'] == 'D',
    matches['result'] == 'A',
]
choices = [matches['p_home'], matches['p_draw'], matches['p_away']]
matches['p_actual'] = np.select(conditions, choices)

matches[['home', 'away', 'result', 'p_home', 'p_draw', 'p_away', 'p_actual']].head()

,home,away,result,p_home,p_draw,p_away,p_actual
0,Valencia,Getafe,H,0.364987,0.320122,0.314891,0.364987
1,Cadiz,Levante,D,0.344140,0.296195,0.359665,0.296195
2,Mallorca,Betis,D,0.282783,0.290518,0.426699,0.290518
3,Alaves,Real Madrid,A,0.135162,0.209397,0.655441,0.655441
4,Osasuna,Espanol,D,0.417659,0.293386,0.288955,0.293386


In [ ]:
market_log_loss = np.log(matches['p_actual']).mean() * -1
print('log loss:', market_log_loss)

log loss: 0.9668525925350964


In [ ]:
conditions = [
    matches['result'] == 'H',
    matches['result'] == 'D',
    matches['result'] == 'A',
]
choices = [0.489, 0.245,0.266]
dumb_baseline = np.select(conditions, choices)
print(dumb_baseline)
log_dumb_baseline = np.log(dumb_baseline).mean() * -1
print('log loss of dumb baseline:', log_dumb_baseline)

[0.489 0.245 0.245 ... 0.266 0.245 0.489]
log loss of dumb baseline: 1.067294561674863


In [ ]:
# Elo ratings: one strength number per team, updated after every match.
START = 1500      # every team's first rating
K = 20            # how much one result moves a rating
HOME_ADV = 60     # rating bonus for playing at home

ratings = {}            # team -> current rating
elo_home_before = []    # home team's rating BEFORE each match (the feature)
elo_away_before = []    # away team's rating BEFORE each match

for row in matches.itertuples():          # already in date order
    r_home = ratings.get(row.home, START)
    r_away = ratings.get(row.away, START)

    # 1. Record the ratings before the match: no peeking at the result
    elo_home_before.append(r_home)
    elo_away_before.append(r_away)

    # 2. What Elo expected, and what actually happened (home team's view)
    expected_home = 1 / (1 + 10 ** ((r_away - (r_home + HOME_ADV)) / 400))
    actual_home = {'H': 1.0, 'D': 0.5, 'A': 0.0}[row.result]

    # 3. Update: the home team gains what the away team loses
    change = K * (actual_home - expected_home)
    ratings[row.home] = r_home + change
    ratings[row.away] = r_away - change

matches['elo_home'] = elo_home_before
matches['elo_away'] = elo_away_before
matches[['date', 'home', 'away', 'result', 'elo_home', 'elo_away']].tail()

,date,home,away,result,elo_home,elo_away
1895,2026-05-23,Celta,Sevilla,H,1531.574312,1468.078831
1896,2026-05-23,Betis,Levante,H,1587.414534,1482.544965
1897,2026-05-23,Alaves,Vallecano,A,1490.710064,1522.611926
1898,2026-05-23,Espanol,Sociedad,D,1464.168448,1497.442747
1899,2026-05-24,Villarreal,Ath Madrid,H,1619.156232,1636.578756


In [ ]:
# Check: final ratings at the end of the season, strongest first
final_elo = pd.Series(ratings).sort_values(ascending=False).round(0)
print(final_elo.head(5))
print('...')
print(final_elo.tail(3))

Barcelona      1774.0
Real Madrid    1740.0
Villarreal     1628.0
Ath Madrid     1628.0
Betis          1593.0
dtype: float64
...
Almeria       1384.0
Granada       1354.0
Valladolid    1315.0
dtype: float64


In [ ]:
# Form: points each team earned in its last 5 matches BEFORE this one (win 3, draw 1, loss 0; max 15)
FORM_GAMES = 5
HOME_POINTS = {'H': 3, 'D': 1, 'A': 0}
AWAY_POINTS = {'H': 0, 'D': 1, 'A': 3}

recent = {}       # team -> list of points from every match so far, oldest first
form_home = []
form_away = []

for row in matches.itertuples():          # date order
    home_hist = recent.setdefault(row.home, [])   # new team -> empty list, stored in the dict
    away_hist = recent.setdefault(row.away, [])

    # 1. Record form before the match: no peeking at the result
    form_home.append(sum(home_hist[-FORM_GAMES:]))
    form_away.append(sum(away_hist[-FORM_GAMES:]))

    # 2. Then add this match's points to each team's history
    home_hist.append(HOME_POINTS[row.result])
    away_hist.append(AWAY_POINTS[row.result])

matches['form_home'] = form_home
matches['form_away'] = form_away
matches[['date', 'home', 'away', 'result', 'form_home', 'form_away']].tail()

,date,home,away,result,form_home,form_away
1895,2026-05-23,Celta,Sevilla,H,7,9
1896,2026-05-23,Betis,Levante,H,8,10
1897,2026-05-23,Alaves,Vallecano,A,10,9
1898,2026-05-23,Espanol,Sociedad,D,7,3
1899,2026-05-24,Villarreal,Ath Madrid,H,7,12


In [ ]:
# Check: form must be between 0 and 15, and better form should mean more home wins
print('range:', matches['form_home'].min(), '-', matches['form_home'].max())
matches.groupby(pd.cut(matches['form_home'], [-1, 4, 9, 15]))['result'].value_counts(normalize=True).unstack().round(2)

range: 0 - 15


result,A,D,H
form_home,,,
"(-1, 4]",0.32,0.32,0.37
"(4, 9]",0.30,0.25,0.45
"(9, 15]",0.20,0.21,0.59


In [ ]:
# Features (X) and target (y)
matches['elo_diff'] = matches['elo_home'] - matches['elo_away']   # who's stronger, in one number
FEATURES = ['elo_diff', 'form_home', 'form_away']
TARGET = 'result'

# Split by TIME, not randomly: learn from older seasons, test on the newest one.
# 2021/22 is left out because Elo was still warming up from 1500 that season.
TRAIN_SEASONS = ['2223', '2324', '2425']
TEST_SEASON = '2526'

train = matches[matches['season'].isin(TRAIN_SEASONS)]
test = matches[matches['season'] == TEST_SEASON]

X_train, y_train = train[FEATURES], train[TARGET]
X_test, y_test = test[FEATURES], test[TARGET]

print('train:', len(train), 'matches', train['date'].min().date(), '->', train['date'].max().date())
print('test: ', len(test), 'matches', test['date'].min().date(), '->', test['date'].max().date())

train: 1140 matches 2022-08-12 -> 2025-05-25
test:  380 matches 2025-08-15 -> 2026-05-24


In [ ]:
model  = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
model.fit(X_train, y_train)
predict_probab = model.predict_proba(X_test)
print(model.classes_)
print(log_loss(y_test, predict_probab))
market_log_loss = np.log(matches[matches['season'] == TEST_SEASON]['p_actual']).mean() * -1
print('log loss of market:', market_log_loss)

['A' 'D' 'H']
0.9823113630631614
log loss of market: 0.9641034157339412
